In [1]:
import pandas as pd
import numpy as np

In [2]:
test = pd.read_csv("test.csv")

In [3]:
test.describe()

,id,age,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time
count,296302.000000,279164.000000,263514.000000,248905.000000,236882.000000,268525.000000,273847.000000,262081.000000,270597.000000,245605.000000
mean,839519.500000,26.608689,7.640771,2.471224,1.457668,2.365962,6.801704,145.748066,102.656216,9.474575
std,85535.164068,5.153359,2.725921,1.318143,0.934727,1.257023,1.236080,66.014554,48.186200,2.858384
min,691369.000000,18.000000,0.500000,0.000000,0.000000,0.000000,4.500000,20.000000,15.000000,0.510000
25%,765444.250000,22.000000,5.460000,1.450000,0.690000,1.360000,5.770000,93.000000,63.000000,7.270000
50%,839519.500000,27.000000,7.770000,2.320000,1.330000,2.200000,6.800000,150.000000,104.000000,9.580000
75%,913594.750000,31.000000,9.850000,3.370000,2.090000,3.200000,7.870000,204.000000,145.000000,11.750000
max,987670.000000,35.000000,15.000000,7.850000,4.000000,6.000000,9.000000,250.000000,180.000000,17.560000


In [4]:
test.info()

<class 'pandas.DataFrame'>
RangeIndex: 296302 entries, 0 to 296301
Data columns (total 13 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       296302 non-null  int64  
 1   age                      279164 non-null  float64
 2   daily_screen_time_hours  263514 non-null  float64
 3   social_media_hours       248905 non-null  float64
 4   gaming_hours             236882 non-null  float64
 5   work_study_hours         268525 non-null  float64
 6   sleep_hours              273847 non-null  float64
 7   notifications_per_day    262081 non-null  float64
 8   app_opens_per_day        270597 non-null  float64
 9   weekend_screen_time      245605 non-null  float64
 10  gender                   282090 non-null  str    
 11  stress_level             276676 non-null  str    
 12  academic_work_impact     270581 non-null  str    
dtypes: float64(9), int64(1), str(3)
memory usage: 29.4 MB


In [5]:
test.head()

,id,age,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,gender,stress_level,academic_work_impact
0,691369,30.0,9.34,NaN,0.77,4.09,7.15,153.0,16.0,NaN,Other,Medium,Yes
1,691370,NaN,NaN,1.91,NaN,1.20,8.67,239.0,148.0,10.68,Female,High,No
2,691371,26.0,8.48,3.64,NaN,3.39,7.47,106.0,123.0,NaN,Male,NaN,No
3,691372,20.0,8.37,2.99,1.69,2.53,5.45,178.0,55.0,9.88,Male,High,Yes
4,691373,25.0,8.25,4.02,1.21,2.32,8.46,63.0,86.0,10.72,Male,Low,No


In [6]:
missing_count = test.isnull().sum()

missing_summary = pd.DataFrame({
    "missing_count": missing_count,
    "missing_percentage": (missing_count / len(test) * 100).round(2)
}).sort_values("missing_count", ascending=False)

display(missing_summary)

,missing_count,missing_percentage
gaming_hours,59420,20.05
weekend_screen_time,50697,17.11
social_media_hours,47397,16.00
notifications_per_day,34221,11.55
daily_screen_time_hours,32788,11.07
work_study_hours,27777,9.37
academic_work_impact,25721,8.68
app_opens_per_day,25705,8.68
sleep_hours,22455,7.58
stress_level,19626,6.62


In [7]:
total_duplicate_rows = test.duplicated().sum()

print(f"Jumlah baris duplikat penuh: {total_duplicate_rows}")

duplicate_rows = test[test.duplicated(keep=False)].sort_values(
    by=test.columns.tolist()
)

display(duplicate_rows)

Jumlah baris duplikat penuh: 0


,id,age,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,gender,stress_level,academic_work_impact


In [8]:
unique_summary = pd.DataFrame({
    "unique_count": test.nunique(dropna=False),
    "dtype": test.dtypes.astype(str),
    "sample_values": [
        test[col].drop_duplicates().head(5).tolist()
        for col in test.columns
    ]
}).sort_values("unique_count")

display(unique_summary)

,unique_count,dtype,sample_values
academic_work_impact,3,str,"[Yes, No, nan]"
stress_level,4,str,"[Medium, High, nan, Low]"
gender,4,str,"[Other, Female, Male, nan]"
age,19,float64,"[30.0, nan, 26.0, 20.0, 25.0]"
app_opens_per_day,167,float64,"[16.0, 148.0, 123.0, 55.0, 86.0]"
notifications_per_day,232,float64,"[153.0, 239.0, 106.0, 178.0, 63.0]"
gaming_hours,402,float64,"[0.77, nan, 1.69, 1.21, 1.19]"
sleep_hours,452,float64,"[7.15, 8.67, 7.47, 5.45, 8.46]"
work_study_hours,602,float64,"[4.09, 1.2, 3.39, 2.53, 2.32]"
social_media_hours,704,float64,"[nan, 1.91, 3.64, 2.99, 4.02]"


In [9]:
# 1. Isi missing value semua kolom numerik dengan median,
#    kecuali stress_level, gender, addicted_label, dan id

excluded_cols = ["stress_level", "gender", "id"]

median_cols = [
    col for col in test.columns
    if col not in excluded_cols and pd.api.types.is_numeric_dtype(test[col])
]

test[median_cols] = test[median_cols].fillna(test[median_cols].median())

In [ ]:
# 2. Ubah gender yang missing/NaN/null menjadi kategori "other"

test["gender"] = (
    test["gender"]
    .astype("string")
    .str.strip()
    .str.lower()
    .replace({"nan": pd.NA, "null": pd.NA, "": pd.NA})
    .fillna("other")
)

KeyboardInterrupt: 

In [ ]:
# 3. Ubah stress_level menjadi ordinal:
#    low = 0, medium = 1, high = 2

stress_mapping = {
    "low": 0,
    "medium": 1,
    "high": 2
}

test["stress_level"] = (
    test["stress_level"]
    .astype("string")
    .str.strip()
    .str.lower()
    .replace({"nan": pd.NA, "null": pd.NA, "": pd.NA})
    .map(stress_mapping)
)

In [ ]:
# 4. Isi stress_level yang missing dengan nilai modus

stress_mode = test["stress_level"].mode()[0]

test["stress_level"] = (
    test["stress_level"]
    .fillna(stress_mode)
    .astype(int)
)

In [ ]:
# Ubah missing/NaN/null pada academic_work_impact menjadi kategori "other"

test["academic_work_impact"] = (
    test["academic_work_impact"]
    .astype("string")
    .str.strip()
    .str.lower()
    .replace({"nan": pd.NA, "null": pd.NA, "": pd.NA})
    .fillna("other")
)

In [ ]:
missing_count = test.isnull().sum()

missing_summary = pd.DataFrame({
    "missing_count": missing_count,
    "missing_percentage": (missing_count / len(test) * 100).round(2)
}).sort_values("missing_count", ascending=False)

display(missing_summary)

,missing_count,missing_percentage
id,0,0.0
age,0,0.0
daily_screen_time_hours,0,0.0
social_media_hours,0,0.0
gaming_hours,0,0.0
work_study_hours,0,0.0
sleep_hours,0,0.0
notifications_per_day,0,0.0
app_opens_per_day,0,0.0
weekend_screen_time,0,0.0


In [ ]:
test_cleaned = test.copy()

In [ ]:
test_cleaned.to_csv("test_cleaned.csv", index=False)

: 